Build Regression Model

Planning to remove late_canc_reason, from_actual_arrival_min , from_actual_departure_min , to_actual_arrival_min,  to_actual_departure_min, propagated_delay_min, end_segment_delay_min, cum_delay_min , prev_train_leg_delay_min , prev_train_cum_delay_min , prev_train_end_segment_delay_min , historical_avg_propagated_delay_min , historical_avg_cum_delay_min , historical_avg_arrival_delay_min , delay_historical_difference


---

Removing this because they will result in the model overfitting the training dataset as these information wont be availabvle

From arrival delay col dodf["delayed"] = (df["arrival_delay_min"] >= 10).astype(int)

In [319]:
import pandas as pd
import plotly.express as px
import numpy as np

In [320]:
pd.set_option('display.float_format', '{:.2f}'.format)
df = pd.read_csv("Datasets/final-segment-dataset-2025.csv")

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11195 entries, 0 to 11194
Data columns (total 38 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   rid                                  11195 non-null  float64
 1   from_station                         11195 non-null  object 
 2   to_station                           11195 non-null  object 
 3   late_canc_reason                     3721 non-null   float64
 4   total_stops_in_run                   11195 non-null  int64  
 5   remaining_stops_after_this           11195 non-null  int64  
 6   date_of_service                      11195 non-null  object 
 7   month                                11195 non-null  int64  
 8   day_of_week                          11195 non-null  int64  
 9   day_of_month                         11195 non-null  int64  
 10  from_sched_arrival_min               11195 non-null  int64  
 11  from_actual_arrival_min     

In [321]:
df.shape

(11195, 38)

In [322]:
#numerical columns
df.describe()

,rid,late_canc_reason,total_stops_in_run,remaining_stops_after_this,month,day_of_week,day_of_month,from_sched_arrival_min,from_actual_arrival_min,from_sched_departure_min,...,historical_avg_leg_duration_min,historical_avg_cum_delay_min,historical_avg_arrival_delay_min,delay_historical_difference,from_temp,from_precip,from_wind,to_temp,to_precip,to_wind
count,11195.00,3721.00,11195.00,11195.00,11195.00,11195.00,11195.00,11195.00,11195.00,11195.00,...,11136.00,11136.00,11136.00,11136.00,11195.00,11195.00,11195.00,11195.00,11195.00,11195.00
mean,202506392764627.06,765.55,9.16,4.58,6.39,1.99,15.60,1042.49,1033.37,1187.20,...,28.82,-187.20,-61.54,41.83,11.40,0.06,12.28,10.77,0.06,12.24
std,3394574883.53,134.35,2.30,2.69,3.39,1.42,8.70,420.57,438.03,105.65,...,27.15,126.15,35.68,143.14,6.69,0.35,6.92,6.32,0.36,7.35
min,202501000000000.00,501.00,4.00,1.00,1.00,0.00,1.00,0.00,0.00,0.00,...,-115.86,-1069.00,-189.44,-1265.00,-26.80,0.00,0.00,-26.80,0.00,0.00
25%,202503000000000.00,650.00,6.00,2.00,3.00,1.00,8.00,1112.00,1113.00,1127.00,...,13.20,-253.16,-76.52,32.57,6.60,0.00,7.20,6.30,0.00,6.80
50%,202506000000000.00,828.00,10.00,4.00,6.00,2.00,16.00,1187.00,1195.00,1194.00,...,23.37,-179.30,-60.26,53.84,11.60,0.00,11.30,11.00,0.00,11.00
75%,202509000000000.00,887.00,11.00,7.00,9.00,3.00,23.00,1257.00,1262.00,1259.00,...,37.04,-89.43,-30.94,67.59,15.80,0.00,16.00,15.00,0.00,16.20
max,202512000000000.00,921.00,14.00,13.00,12.00,4.00,31.00,1391.00,1438.00,1391.00,...,1195.00,132.00,66.00,1133.00,32.40,7.20,60.10,31.50,7.20,61.20


In [ ]:
df['delay_gained_in_segment'] = df['arrival_delay_min'] - df['propagated_delay_min']

features = ["from_temp", "to_temp", "from_precip", "to_precip", "from_wind", "to_wind", "prev_train_leg_delay_min",
"prev_train_cum_delay_min",
"prev_train_segment_duration_min",
"prev_train_end_segment_delay_min",
"historical_avg_propagated_delay_min",
"historical_avg_leg_duration_min",
"historical_avg_cum_delay_min",
"historical_avg_arrival_delay_min",
"to_sched_arrival_min",
"to_sched_departure_min",
"day_of_week"]



In [340]:
# Potential features to use


for feature in features:
    outlier_identifier = px.scatter(df, x=f"{feature}", y="delay_gained_in_segment")
    outlier_identifier.show()

In [ ]:
#Import scikit-learn metrics module for classification and regression performance metrics
from sklearn import metrics

In [ ]:
#Splitting into X and Y

columns_to_drop = ["delay_historical_difference", "leg_duration_min",
                   "cum_delay_min", "end_segment_delay_min", "to_actual_arrival_min", "to_actual_departure_min", 
                   "from_actual_departure_min", "from_actual_arrival_min", "date_of_service", "late_canc_reason", "from_station", "to_station",
                   "rid", "propagated_delay_min", "arrival_delay_min", "delay_gained_in_segment"]

X = df.drop(columns=columns_to_drop)
y = df["delay_gained_in_segment"] # Target Variable



In [ ]:
from sklearn.model_selection import train_test_split

# Split the dataset in 80% Training and 20% Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)
X_train = X_train.fillna(X_train.median())
X_test = X_test.fillna(X_train.median())

In [328]:
from sklearn.preprocessing import RobustScaler
scaler = RobustScaler()
scaler = scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

In [329]:
print('Whole Data shape', df.shape)
print('X_train shape', X_train.shape)
print('X_test shape', X_test.shape)


Whole Data shape (11195, 39)
X_train shape (8956, 23)
X_test shape (2239, 23)


In [ ]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import plotly.graph_objects as go

def evaluate_model(model, model_name, X_train, X_test, y_train, y_test):

  """
  Function evaluates the model by making predictions, printing the MAE, MSE and R2
  

  Returns dataframe of metrics


  """

  y_pred = model.predict(X_test) # Model predictions on test data 


  table = pd.DataFrame({'Model': [f"{model_name}"],
                        'Mean Absolute Error': [round(mean_absolute_error(y_test, y_pred), 2)], # Regression metrics
                        'Mean Sqaured Error': [round(mean_squared_error(y_test, y_pred),2)],
                        'R2': [round(r2_score(y_test, y_pred),2)]})

  return table

In [331]:
from sklearn.tree import DecisionTreeRegressor


# Build a fully grown decision tree 
dt = DecisionTreeRegressor(random_state=123)
dt.fit(X_train, y_train)

#To make predictions on the test set, ues the predict method:
y_pred_clf = dt.predict(X_test)


# Prints metrics for evaluation

dt_metrics = evaluate_model(dt, "Decision Tree", X_train, X_test, y_train, y_test)

In [332]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()

lr.fit(X_train, y_train)

lr_metrics =evaluate_model(lr, "Linear Regression", X_train, X_test, y_train, y_test)


In [333]:
# K-Nearest Neighbors Regression
from sklearn.neighbors import KNeighborsRegressor
import numpy as np

# Build a fully grown decision tree clf
knn = KNeighborsRegressor(n_neighbors=int(np.sqrt(len(df))))
knn.fit(X_train, y_train)

# Prints metrics for evaluation

knn_metrics =evaluate_model(knn, "K-Nearest Neighbors", X_train, X_test, y_train, y_test)


In [334]:
# XGBoost Regression
from xgboost import XGBRegressor

# Build a fully grown decision tree clf
xgb = XGBRegressor()
xgb.fit(X_train, y_train)

# Prints metrics for evaluation

xgb_metrics = evaluate_model(xgb, "XGBoost", X_train, X_test, y_train, y_test)

In [ ]:
# LightGBM Regression
from lightgbm import LGBMRegressor

# Build a fully grown decision tree clf
gbm = LGBMRegressor()
gbm.fit(X_train, y_train)


# Prints metrics for evaluation

gbm_metrics = evaluate_model(gbm, "LightGBM", X_train, X_test, y_train, y_test)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000359 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2283
[LightGBM] [Info] Number of data points in the train set: 8956, number of used features: 23
[LightGBM] [Info] Start training from score -7.233921


C:\Users\danie\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning:

X does not have valid feature names, but LGBMRegressor was fitted with feature names



In [ ]:
# Concatenate all models' metrics
all_metrics = pd.concat(
    [dt_metrics, knn_metrics, lr_metrics, xgb_metrics, gbm_metrics], 
    ignore_index=True
)

# Sort by Mean Absolute Error (ascending)
all_metrics = all_metrics.sort_values(by='Mean Absolute Error', ascending=True).reset_index(drop=True)


# Create Plotly table
fig_metrics = go.Figure(data=[go.Table(
    header=dict(
        values=list(all_metrics.columns),  # Column names: Model, MAE, MSE, R2
        fill_color='firebrick',
        font=dict(color='white', size=12),
        align='center'
    ),
    cells=dict(
        values=[all_metrics[col] for col in all_metrics.columns],  # Values from each column
        fill_color=[['#f9f9f9', 'white'] * (len(all_metrics)//2 + 1)], 
        align='center'
    )
)])

# Optional layout adjustments
fig_metrics.update_layout(
    title="Regression Model Metrics",
    template='plotly_white',
    width=800,
    height=400
)

    
fig_metrics.update_layout(
    title=dict(
        x=0.5,       # Center of the x-axis
        xanchor='center'  # Anchor the title at the center
    ),)

# Show table
fig_metrics.show()



In [337]:
# For some reason X_train cannot be used for column names unless split again

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)
X_train = X_train.fillna(X_train.median())
X_test = X_test.fillna(X_train.median())
    


feat_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': gbm.feature_importances_
    }).sort_values(by='Importance', ascending=True)  # ascending for horizontal bar chart
        
fig_imp = px.bar(
    feat_df,
    x='Importance',
    y='Feature',
    orientation='h',
    color='Importance',
    color_continuous_scale='Reds',
    title='LightGBM Feature Importance'
)

fig_imp.update_layout(
    yaxis=dict(title='Feature'),
    xaxis=dict(title='Importance'),
    template='plotly_white',
    width=800,
    height=600
)
    
fig_imp.update_layout(
    title=dict(
    x=0.5,       # Center of the x-axis
    xanchor='center'  # Anchor the title at the center
    ),)
    
fig_imp.show()